In [124]:
import os,glob,re,json
from lxml import etree

In [ ]:
parser = etree.XMLParser(remove_blank_text=True, resolve_entities=True)

doec_location = '../DOEC/xml-corpus/'
echoe_location = '../echoe/xml/'

def regsplit(delimiters, string, maxsplit=0):
    import re
    regex_pattern = '|'.join(map(re.escape, delimiters))
    return re.split(regex_pattern, string, maxsplit)

def normalize(target):
    matrix = {
        '&': 'and',
        'ð': 'þ',
        'ę': 'æ',
        'á': 'a',
        'ǽ': 'æ',
        'é': 'e',
        'í': 'i',
        'ó': 'o',
        'ú': 'u',
        'ý': 'y',
        'ẏ': 'y',
        'ƿ': 'w',
        'ſ': 's',
        '': 's', # descending s
        'v': 'u',
        'j': 'i',
        '⁊': 'and',
        '·': '',
        ' ': '',
        '\n': ''

        }
    for k,v in matrix.items():
        target = target.lower().replace(k, v)
    return target

# TODO: this is currently not discarding anything?
def simplify(branch):
    discard = ['abbr', 'am', 'del', 'sic', 'note']
    query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
    for hit in branch.iter(query):
        hit.getparent().remove(hit)

def load_from_toc(source_dict, doec_location):
    corpus = dict()
    for ref,title in source_dict.items():
        segments = []
        file = doec_location + ref + '.xml'
        tree = etree.parse(file, parser=parser)
        root = tree.getroot()
        text = root.find('.//text')
        for segment in text.iter('s'):
            plaintext = regsplit(':;', normalize(etree.tostring(segment, method='text', encoding='unicode')))
            segments = segments + plaintext
        corpus[title] = segments
    return corpus

def load_echoe(echoe_location):
    corpus = dict()
    for doc in sorted(glob.glob(echoe_location + '*.xml')):
        identifier = os.path.basename(doc).replace('.xml', '')
        segments = []
        tree = etree.parse(doc, parser=parser)
        root = tree.getroot()
        text = root.find('.//{http://www.tei-c.org/ns/1.0}text')
        simplify(text)
        for segment in text.iter('{http://www.tei-c.org/ns/1.0}s'):
            tokens = []
            for token in segment.iter('{http://www.tei-c.org/ns/1.0}w'):
            # If a word element is marked as the last part of a word, add its text content to the preceding token:
                if token.get('part') == 'F':
                    position = len(tokens)-1
                    tokens[position] = tokens[position] + normalize(etree.tostring(token, method='text', encoding='unicode'))
                else:
                    tokens.append(normalize(etree.tostring(token, method='text', encoding='unicode')))
            segments.append(tokens)
        corpus[identifier] = segments
    return(corpus)


In [126]:
with open('../aelfric/doec-aelfric-toc.json', 'r') as json_file:
    aelfric_toc = json.load(json_file)
    



In [127]:
aelfric = load_from_toc(aelfric_toc, doec_location)

In [128]:
echoe = load_echoe(echoe_location)

In [129]:
echoe['018.40']

[['mage',
  'we',
  'gyt',
  'her',
  'gehyran',
  'men',
  'þa',
  'leofestan',
  'eowre',
  'sawle',
  'þearfe',
  'gif',
  'ge',
  'me',
  'hlystan',
  'wyllaþ',
  'and',
  'on',
  'eowre',
  'heortan',
  'þas',
  'halgan',
  'lare',
  'underniman'],
 ['swa',
  'swa',
  'sc̄ssanctus',
  'augustinꝰus',
  'hit',
  'ærest',
  'on',
  'bocūm',
  'awrat',
  'and',
  'þus',
  'cw̄æþ',
  'ic',
  'eow',
  'lære',
  'ealle',
  'ægþer',
  'ge',
  'broþra',
  'ge',
  'swystra',
  'ge',
  'weras',
  'ge',
  'wif',
  'geonge',
  'men',
  'and',
  'ealde',
  '\uf149þæt',
  'ge',
  'eow',
  'scyldan',
  'and',
  'wæpnigan',
  'wiþ',
  'þam',
  'wyrrestan',
  'feonde'],
 ['forþām',
  'þe',
  'he',
  'is',
  'on',
  'us',
  'weard',
  'swa',
  'þe',
  'sceaþa',
  'ure',
  'godan',
  'weorc',
  'to',
  'forstelenne',
  'and',
  'he',
  '\uf149þæt',
  'mid',
  'yfelūm',
  'geþance',
  'and',
  'gediht',
  'eall'],
 ['we',
  'secgaþ',
  'eac',
  'to',
  'soþan',
  '\uf149þæt',
  'hit',
  'is',
  'yf